In [32]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler


In [14]:
CSV_PATH = "raw_loan_dataset.csv"
OUT_PATH = ".clean_loan_dataset.csv"

In [16]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
df.head()

print("\n=== INITIAL HEAD ===")
print(df.head(10))

Shape: (103, 7)

=== INITIAL HEAD ===
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   
5      NaN        754.0              3.6      47800           Yes   
6   106070        665.0             14.6      48000             N   
7    35458        599.0             21.7    $10,300            No   
8    73520        661.0              NaN      17200           Yes   
9    57087        563.0             11.8      13400           Yes   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  
5               

In [6]:
print("\n=== INITIAL INFO ===")
print(df.info())


=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB
None


In [7]:
print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())


=== INITIAL MISSING VALUES ===
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


In [17]:
# Clean currency formatting
df["Income"] = df["Income"].replace(r"[\$,]", "", regex=True).astype(float)
df["LoanAmount"] = df["LoanAmount"].replace(r"[\$,]", "", regex=True).astype(float)

In [19]:
# Normalize categorical typos BEFORE imputation
yes_no_map = {
    "yes": "Yes", "y": "Yes", "yse": "Yes", "1": "Yes", "approved": "Yes",
    "no": "No", "n": "No", "0": "No", "rejected": "No",
}

df["HasCollateral"] = df["HasCollateral"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["PreviousDefaults"] = df["PreviousDefaults"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
df["Approved"] = df["Approved"].astype(str).str.strip().str.lower().replace(yes_no_map).replace({"nan": np.nan})
print("Data after cleaning and normalizing:")
print(df.head())


Data after cleaning and normalizing:
     Income  CreditScore  EmploymentYears  LoanAmount HasCollateral  \
0  108810.0        537.0              1.1     25800.0           Yes   
1   96482.0        524.0             15.0     11200.0           Yes   
2   28478.0          NaN              5.4     12100.0            No   
3   25851.0        561.0             17.6      7000.0           Yes   
4   38396.0        527.0              9.8     10700.0            No   

  PreviousDefaults Approved  
0               No       No  
1               No      Yes  
2               No      Yes  
3               No      Yes  
4               No      Yes  


In [20]:
#Impute missing values
df["Income"] = df["Income"].fillna(df["Income"].median())
df["CreditScore"] = df["CreditScore"].fillna(df["CreditScore"].median())
df["EmploymentYears"] = df["EmploymentYears"].fillna(df["EmploymentYears"].median())
df["LoanAmount"] = df["LoanAmount"].fillna(df["LoanAmount"].median())
df["HasCollateral"] = df["HasCollateral"].fillna(df["HasCollateral"].mode()[0])
df["PreviousDefaults"] = df["PreviousDefaults"].fillna(df["PreviousDefaults"].mode()[0])
print ("====missing vlue====")
print(df.isnull().sum())


====missing vlue====
Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64


In [11]:
# Remove duplicates
before = df.shape[0]
df = df.drop_duplicates()
print(f"\nDropped duplicates: {before} -> {df.shape[0]} rows")


Dropped duplicates: 103 -> 100 rows


In [18]:
# IQR capping on numeric columns
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_income,  high_income  = iqr_fun(df["Income"])
low_credit,  high_credit  = iqr_fun(df["CreditScore"])
low_loan,    high_loan    = iqr_fun(df["LoanAmount"])
low_emp,     high_emp     = iqr_fun(df["EmploymentYears"])

df["Income"]          = df["Income"].clip(lower=low_income,  upper=high_income)
df["CreditScore"]     = df["CreditScore"].clip(lower=low_credit, upper=high_credit)
df["LoanAmount"]      = df["LoanAmount"].clip(lower=low_loan,    upper=high_loan)
df["EmploymentYears"] = df["EmploymentYears"].clip(lower=low_emp,     upper=high_emp)



In [21]:
#  Label encoding (Yes/No → 0/1)
df["Approved"] = df["Approved"].map({"Yes": 1, "No": 0}).astype(int)

print("\n=== CLASS DISTRIBUTION (after label encoding) ===")
print(df["Approved"].value_counts())
print(df["Approved"].value_counts(normalize=True).round(3))
df['Approved'].value_counts()


=== CLASS DISTRIBUTION (after label encoding) ===
Approved
1    68
0    35
Name: count, dtype: int64
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


Approved
1    68
0    35
Name: count, dtype: int64

In [22]:
# Class balance check
class_ratio = df["Approved"].value_counts(normalize=True).min()
if class_ratio < 0.30:
    print("\nWarning: severely imbalanced classes — consider balancing techniques (L3).")
else:
    print("\nClass balance OK for baseline Accuracy (both classes well represented).")

df["HasCollateral"] = df["HasCollateral"].map({"Yes": 1, "No": 0}).astype(int)
df["PreviousDefaults"] = df["PreviousDefaults"].map({"Yes": 1, "No": 0}).astype(int)


Class balance OK for baseline Accuracy (both classes well represented).


In [23]:
# Feature engineering (no leakage from label)
df['DebtToIncome'] = df['LoanAmount'] / df['Income']

df['IncomePerYearEmployed'] = df['Income'] / df['EmploymentYears'].replace(0, np.nan)
df['IncomePerYearEmployed'] = df['IncomePerYearEmployed'].fillna(df['IncomePerYearEmployed'].median())

df[['Income', 'LoanAmount', 'EmploymentYears', 'DebtToIncome', 'IncomePerYearEmployed']].head()


,Income,LoanAmount,EmploymentYears,DebtToIncome,IncomePerYearEmployed
0,108810.0,25800.0,1.1,0.237111,98918.181818
1,96482.0,11200.0,15.0,0.116084,6432.133333
2,28478.0,12100.0,5.4,0.424889,5273.703704
3,25851.0,7000.0,17.6,0.270783,1468.806818
4,38396.0,10700.0,9.8,0.278675,3917.959184


In [27]:
# StandardScaler on numeric features
binary_cols = ["Approved", "HasCollateral", "PreviousDefaults"]

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

scale_cols = [col for col in numeric_cols if col not in binary_cols]

scaler = RobustScaler()

df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("RobustScaler applied successfully!")
print("Scaled columns:", scale_cols)

df.head()

RobustScaler applied successfully!
Scaled columns: ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'DebtToIncome', 'IncomePerYearEmployed']


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.809369,-0.655844,-0.978632,0.245192,1,0,0,-0.443524,13.641865
1,0.555798,-0.740260,0.209402,-0.456731,1,0,1,-0.921977,0.111474
2,-0.842958,0.000000,-0.611111,-0.413462,0,0,1,0.298819,-0.058000
3,-0.896992,-0.500000,0.431624,-0.658654,1,0,1,-0.310409,-0.614644
4,-0.638957,-0.720779,-0.235043,-0.480769,0,0,1,-0.279209,-0.256341


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 103 non-null    float64
 1   CreditScore            103 non-null    float64
 2   EmploymentYears        103 non-null    float64
 3   LoanAmount             103 non-null    float64
 4   HasCollateral          103 non-null    int64  
 5   PreviousDefaults       103 non-null    int64  
 6   Approved               103 non-null    int64  
 7   DebtToIncome           103 non-null    float64
 8   IncomePerYearEmployed  103 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.4 KB


In [29]:
print(df.isnull().sum())

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


In [30]:
print("Missing values remaining:")
print(df.isnull().sum())
df.head()

Missing values remaining:
Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64


,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.809369,-0.655844,-0.978632,0.245192,1,0,0,-0.443524,13.641865
1,0.555798,-0.740260,0.209402,-0.456731,1,0,1,-0.921977,0.111474
2,-0.842958,0.000000,-0.611111,-0.413462,0,0,1,0.298819,-0.058000
3,-0.896992,-0.500000,0.431624,-0.658654,1,0,1,-0.310409,-0.614644
4,-0.638957,-0.720779,-0.235043,-0.480769,0,0,1,-0.279209,-0.256341


In [31]:
df.to_csv('clean_loan_dataset.csv', index=False)
print("Saved clean_loan_dataset.csv")

Saved clean_loan_dataset.csv
